# YuE2 Studio 5.0.0 — Cloudflare Runtime

**This launches the real YuE2 Studio web interface and its Rust backend. It is not a ComfyUI workflow.**

## Kaggle settings

- Accelerator: **GPU T4 ×2** (or another supported NVIDIA GPU)
- Internet: **ON**
- Add Input: attach the saved output from `YuE2_Studio_Builder.ipynb`.

## Start

Run the single code cell. It will:
1. Locate and verify the builder bundle.
2. Install the small Linux runtime dependencies and Node 22.
3. Clone the upstream `v5.0.0` Studio source and install its frontend dependencies.
4. Start the real Rust music service, the YuE2 CUDA/Vulkan engine and the upstream Vite interface.
5. Open a Cloudflare Quick Tunnel and print the public Studio link.

Open the printed `trycloudflare.com` address. Keep the Kaggle cell running.

## First use

Use the Studio's own setup / Models screens to download the model sets and optional engines you want. The builder does not pre-download multi-gigabyte model weights. Your songs and library data are stored under `/kaggle/working/yue2-studio-data` for this Kaggle session; export or save them before the session ends.

## Important

This is a headless browser-hosted run of the upstream application, not a native Windows desktop build. The upstream UI and Rust service are used; the Vite development server proxies the Studio API to its Rust backend. Optional engines and tools may be platform- or model-licence-dependent. The Studio application is MIT; that does not make every model weight or engine commercially licensed.

Logs: `music-server.log`, `studio-ui.log`, `cloudflared.log` in `/kaggle/working`.

In [ ]:
import os, sys, json, shutil, subprocess, time, re, urllib.request
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
DATA = WORK / "yue2-studio-data"
SHARED = WORK / "timoncool-shared"
REPO = WORK / "YuE2-Studio"
DATA.mkdir(parents=True, exist_ok=True)
SHARED.mkdir(parents=True, exist_ok=True)

# Find the private builder output attached through Kaggle → Add Input → Notebook/Dataset.
manifests = list(INPUT_ROOT.rglob("manifest.json"))
bundle = None
for p in manifests:
    try:
        info = json.loads(p.read_text())
        if info.get("studio_version") == "5.0.0" and info.get("tag") == "v5.0.0":
            bundle = p.parent
            break
    except Exception:
        pass
if bundle is None:
    raise FileNotFoundError("Attach the saved output from YuE2_Studio_Builder.ipynb as a Kaggle input. No v5.0.0 bundle/manifest.json was found.")

required = [bundle / "music-server", bundle / "cloudflared", bundle / "yue2-cpp/yue-server"]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Builder output is incomplete: " + ", ".join(missing))

# Install runtime libraries only; the Studio service and CUDA engine were built in notebook 1.
subprocess.run(["bash", "-lc", "apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2 libvulkan1 libgtk-3-0 libayatana-appindicator3-1 ffmpeg"], check=True)

# Node 22 is used for the upstream Vite UI. The Rust service and YuE2 CUDA engine come from the bundle.
node_dir = WORK / "node-v22"
if not (node_dir / "bin/node").exists():
    node_archive = WORK / "node-v22-linux-x64.tar.xz"
    urllib.request.urlretrieve("https://nodejs.org/dist/latest-v22.x/node-v22.16.0-linux-x64.tar.xz", node_archive)
    node_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(["tar", "-xJf", str(node_archive), "-C", str(node_dir), "--strip-components=1"], check=True)
env = os.environ.copy()
env["PATH"] = f"{node_dir}/bin:/root/.cargo/bin:/usr/local/bin:/usr/bin:/bin"
env["YUE_STUDIO_PORT"] = "8791"
env["YUE_STUDIO_DATA_ROOT"] = str(DATA)
env["TIMONCOOL_SHARED_DIR"] = str(SHARED)
env["YUE_ENGINE_BIN"] = str(bundle / "yue2-cpp/yue-server")
env["YUE_ENGINE_ROOT"] = str(bundle / "yue2-cpp")
env["CUDA_VISIBLE_DEVICES"] = "0,1"
env["DO_NOT_TRACK"] = "1"
env["STUDIO_TELEMETRY"] = "0"
env["PYTHONUNBUFFERED"] = "1"

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "v5.0.0", "https://github.com/timoncool/YuE2-Studio.git", str(REPO)], check=True)
subprocess.run(["npm", "ci", "--no-audit", "--no-fund"], cwd=REPO / "app", env=env, check=True)

server_log = open(WORK / "music-server.log", "w")
server = subprocess.Popen([str(bundle / "music-server")], cwd=REPO, env=env, stdout=server_log, stderr=subprocess.STDOUT)
print("Starting YuE2 Studio Rust service...")
for _ in range(90):
    if server.poll() is not None:
        raise RuntimeError("Studio backend exited. Check /kaggle/working/music-server.log")
    try:
        import urllib.request
        urllib.request.urlopen("http://127.0.0.1:8791/health", timeout=2).read()
        break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError("Studio backend did not become healthy. Check music-server.log")

ui_log = open(WORK / "studio-ui.log", "w")
ui = subprocess.Popen(["npm", "run", "dev", "--", "--host", "0.0.0.0", "--port", "3791", "--strictPort"],
                      cwd=REPO / "app", env=env, stdout=ui_log, stderr=subprocess.STDOUT)
for _ in range(45):
    if ui.poll() is not None:
        raise RuntimeError("Studio UI exited. Check /kaggle/working/studio-ui.log")
    try:
        urllib.request.urlopen("http://127.0.0.1:3791/", timeout=2).read()
        break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError("Studio UI did not become ready. Check studio-ui.log")

# Expose the actual YuE2 Studio web UI; Vite proxies its API routes to the Rust service on 8791.
cloudflared = bundle / "cloudflared"
tunnel_log = open(WORK / "cloudflared.log", "w")
tunnel = subprocess.Popen([str(cloudflared), "tunnel", "--url", "http://127.0.0.1:3791", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
print("\nYuE2 Studio 5.0.0 is starting its Cloudflare Quick Tunnel.")
print("Keep this Kaggle cell running while you use the Studio.\n")
try:
    for line in tunnel.stdout:
        print(line.rstrip(), flush=True)
        tunnel_log.write(line)
        tunnel_log.flush()
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if match:
            print("\nOPEN YOUR YUe2 STUDIO:", match.group(0), "\n", flush=True)
except KeyboardInterrupt:
    print("Stopping Studio processes...")
finally:
    for proc in (tunnel, ui, server):
        if proc and proc.poll() is None:
            proc.terminate()
    print("Studio stopped. Your library and generated files remain in:", DATA)
